# ZTE — the evidence suite

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/victor-iyi/zte/blob/main/notebooks/tbme/zte_tbme.ipynb)

**Upload this notebook on its own.** It clones the repo, reads your Drive, and runs the eight experiments that
together decide what this programme is entitled to claim. It ends with one command that assembles every measured
number beside the brain-free floor it has to clear.

| § | Experiment | What it settles | Trains? | Roughly |
| --- | --- | --- | --- | --- |
| **6** | Granularity ablation | Does sentence, word or token alignment recover anything spelling does not? | optional | minutes to re-read; ~110 GPU-h to re-train |
| **7** | Passage confound | Is cross-subject retrieval semantics, or memorised passages? | no | ~30 min |
| **8** | Decoder reality check | Does generated text carry the EEG, or the LM's prior? | no | ~40 min |
| **9** | Anchor calibration | What does a new reader gain from a few labelled sentences, without retraining? | **no** | ~20 min |
| **10** | Semantic hard negatives | Does punishing length- and piece-matched confusions raise retrieval? | yes | ~4 h |
| **11** | Architecture benchmark | Does the conformer beat EEGNet and DeepConvNet through the identical pipeline? | yes | ~7 h |
| **12** | Physiological interpretability | *When* in the word, and *where* on the scalp, does the readout come from? | no | ~15 min |
| **13** | Twelve-fold LOSO | The population number, mean ± sd, not one lucky holdout | yes | ~110 GPU-h |
| **14** | **The evidence board** | Every claim, its floor, its verdict | no | seconds |

Sections 7, 8, 9, 12 and 14 read checkpoints and cost minutes. **Start there.** Sections 6, 10, 11 and 13 train, and
section 13 is the expensive one.

## How to read a number in this project

Three rules, and none of them is stylistic.

1. **`held_out_retrieval`, never `sentence_retrieval`.** The pooled number is computed over the training subjects
   too, so it rewards memorising the brains you have rather than reaching the one you do not. It inverted the
   champion once already: an arm with a pooled Top-1 of 0.043 scored 4 hits in 700 held out, and an identical re-run
   gave 2.
2. **Every retrieval number is read against a brain-free floor.** On the real 700-sentence gallery the word count
   alone retrieves **53** sentences and the total sub-word piece count **71**. The best encoder this programme has
   trained retrieves **33**. A number quoted without its floor is not evidence, whatever its *p*-value.
3. **Top-*k* is a hit count out of the queries actually scored, with an exact binomial tail — never a bare rate.**
   At chance 1/700, Top-1 expects exactly one hit, so a headline of "0.006 versus 0.001" is three hits and a coin.

Section 14 enforces all three mechanically: a row with no floor renders as `not measured`, and a row whose
confidence interval straddles its floor renders as below it. **Nothing on that board can go green by accident.**

## What is already known, before you run anything

So that no cell below comes as a surprise, and so no result here is oversold:

- The three alignment levels have been measured over twelve folds. **None clears the length-oracle floor**, and
  `token` — the arm most exposed to the spelling channel — is nominally highest. That ordering is the confound
  signature, not a win.
- The parallax transfer matrix already shows a code reaching a never-seen subject reading never-seen sentences at
  rank percentile ~0.95, with NR→SR the *strongest* cell. That is the passage-confound answer, and it is a positive
  result.
- The encoder supplies ~1.7–2.0 bits of sentence identity against the 9.45 a sentence needs. **Expect an honest null
  on generation**, and read section 8 as a control experiment rather than a headline.
- No trained checkpoint has an attentive temporal pool, so section 12 measures *occlusion* rather than attention.
  See its own preamble for why that is the better instrument anyway.

## 1 · Provision the runtime

Installs `uv`, clones or refreshes the repo, and builds the pinned Python 3.14 venv that every `!uv run` below uses.
The kernel you are typing in is Colab's own older interpreter and never imports `zte`.

In [ ]:
%%bash
pip install -q uv
# Work whether this is a fresh runtime (/content), a re-run already inside zte/, or a restored session.
if [ -f pyproject.toml ]; then :
elif [ -d zte/.git ]; then cd zte
else git clone --depth 1 https://github.com/victor-iyi/zte.git --branch main && cd zte
fi
git fetch --depth 1 origin main && git reset --hard FETCH_HEAD
echo "ZTE @ $(git rev-parse --short HEAD): $(git log -1 --pretty=%s)"
uv python install 3.14
uv sync --all-groups

## 2 · Wire the kernel

`colab()` is the only route into the package: it runs one `zte-colab` subcommand in the venv and returns the JSON it
printed. Nothing below computes with ZTE in this kernel — it renders payloads the venv produced.

In [ ]:
import json
import os
import platform
import subprocess
from typing import Any


def colab(command: str, *args: str) -> dict[str, Any]:
    """Runs one `zte-colab` subcommand in the provisioned venv and returns the JSON object it printed.

    This is the notebook's only route into ZTE. The package runs on 3.14 inside the uv venv; this kernel is
    Colab's own older interpreter, so it renders payloads rather than computing them.
    """
    argv = ['uv', 'run', 'zte-colab', command, *args]
    done = subprocess.run(argv, capture_output=True, text=True, check=False)
    if done.returncode != 0:
        raise RuntimeError(f'`{" ".join(argv)}` failed:\n{done.stderr[-3000:]}')

    return json.loads(done.stdout)


# Enter the repo in the notebook kernel, so relative paths and every subprocess resolve. A %%bash `cd` cannot
# do this: it dies with its own shell.
if os.path.isdir('zte') and not os.path.isfile('pyproject.toml'):
    os.chdir('zte')

ENV = colab('env')
os.environ.update(ENV['env'])

try:
    from google.colab import userdata  # type: ignore[import-untyped]

    _hf = userdata.get('HF_TOKEN')
except Exception as exc:  # not on Colab, or the secret is not granted to this notebook
    _hf, _ = None, print(f'HF_TOKEN unavailable ({type(exc).__name__}) - Hub downloads will be unauthenticated.')
if _hf:
    os.environ['HF_TOKEN'] = _hf
    print('HF_TOKEN loaded - authenticated HuggingFace Hub downloads enabled.')

print(f'repo   : {ENV["root"]}')
print(f'venv   : Python {ENV["venv"]["python"]} - zte {ENV["venv"]["zte"]}   <- every `!uv run` command')
print(f'kernel : Python {platform.python_version()}   <- this cell; renders payloads, never imports zte')

## 3 · What hardware did you get

Sections 6, 10, 11 and 13 train and need an accelerator. Everything else reads checkpoints and runs on CPU.

In [ ]:
plan, res = ENV['plan'], ENV['resources']

print(f'backend   : {plan["backend"]}  -  device {plan["device"]}  -  autocast {plan["autocast_dtype"]}')
print(f'resources : {res["ram_gb"]} GB RAM - {res["cpu_count"]} cores - {res["free_disk_gb"]} GB free')
print(f'gpu       : {res.get("gpu") or "none - the training sections need one"}')
if not res.get('gpu'):
    print('\nNo accelerator: Runtime -> Change runtime type -> GPU, or run only the read-only sections.')

## 4 · Drive is the workspace

Every artifact lands in a dated session folder on Drive, so a reclaimed VM costs you nothing. Set `RESUME_DATE` to
an existing folder name to continue a session; leave it `None` to start today's.

In [ ]:
from google.colab import drive  # type: ignore[import-untyped]

drive.mount('/gdrive')

In [ ]:
# Set to an existing folder name (e.g. '2026-08-24') to resume that session; None starts today's.
RESUME_DATE: str | None = None
# 'auto' writes runs to Drive when it is mounted, and to the local disk otherwise. On Colab that is what keeps a
# twelve-fold sweep inside the VM's disk: 54 runs are ~27 GB of checkpoints beside an 11-24 GB bundle.
WRITE_MODE: str = 'auto'
ZTE_DRIVE: str = '/gdrive/My Drive/Sharables/ZTE'

_resume = ('--resume-date', RESUME_DATE) if RESUME_DATE else ()
SESSION = colab('session', '--drive', ZTE_DRIVE, '--write-mode', WRITE_MODE, *_resume)
os.environ.update(SESSION['env'])

RUN_DATE: str = SESSION['run_date']
DATA_DIR: str = SESSION['data_dir']
DRIVE_DIR: str = SESSION['session_dir']
DRIVE_ANALYSIS: str = SESSION['drive_analysis']
LOCAL_RUNS: str = SESSION['local_runs']
OUT_ROOT: str = SESSION['out_root']
PREPARED_LOCAL: str = SESSION['prepared_local']
PREPARED_DRIVE: str = SESSION['prepared_drive']

# Everything this notebook measures lands under one root, which is also the only thing section 14 has to read.
SUITE: str = f'{DRIVE_ANALYSIS}/evidence_suite'

print(f'session   : {RUN_DATE}   ({"resumed" if SESSION["resumed"] else "new"})')
print(f'raw data  : {DATA_DIR}   (present: {SESSION["data_dir_present"]})')
print(f'runs ->   : {OUT_ROOT}')
print(f'suite ->  : {SUITE}')
if not SESSION['data_dir_present']:
    print('\nZuCo is not at that path. Sections that read the corpus will fail until it is.')

### 4a · Helpers this notebook uses everywhere

`audit()` and `show()` are the pair every experiment below ends with: one reads an audit's JSON through the bridge,
the other displays the Markdown that audit's own CLI wrote. Nothing is recomputed in this kernel, so what you read
is what the run recorded.

In [ ]:
import pathlib


def find_runs(*extra: str, headline: bool = False) -> list[dict[str, Any]]:
    """Every run reachable right now - each dated Drive session newest first, then the local disk."""
    flags = ['--headline'] if headline else []

    return colab('runs', '--drive', ZTE_DRIVE, '--experiments', *extra, LOCAL_RUNS, *flags)['runs']


def every_session() -> list[str]:
    """Every dated session's run folder on Drive, newest first - what the analysis sections read across."""
    return colab('runs', '--drive', ZTE_DRIVE)['sessions']


def resolve_ckpt(run_name: str, which: str = 'best') -> str:
    """Finds a run's checkpoint, Drive first, so a fresh VM can audit a session it did not train.

    A missing `best.pt` never falls back to `last.pt`: they are different models, and swapping them silently
    misattributes the number.
    """
    for run in colab('runs', '--drive', ZTE_DRIVE, '--experiments', LOCAL_RUNS, '--run', run_name)['runs']:
        if path := run['checkpoints'][which]:
            print(f'{which}.pt for {run_name}: {"Drive" if run["source"] == "drive" else "local disk"}\n  {path}')
            return path

    raise FileNotFoundError(f'no {which}.pt for {run_name!r} on Drive or locally; train it first.')


def audit(kind: str, directory: str, markdown: bool = True) -> dict[str, Any]:
    """Reads one audit's JSON (and its Markdown) out of a directory, recomputing nothing."""
    flags = [] if markdown else ['--no-markdown']
    payload = colab('audit', '--from', directory, '--kind', kind, *flags)['audits'][kind]
    if not payload['found']:
        print(f'no {kind} artifact at {payload["json_path"]} - run the cell above it first.')

    return payload


def show(payload: dict[str, Any]) -> None:
    """Renders an audit's own Markdown inline, so the notebook shows the report the CLI wrote."""
    from IPython.display import Markdown, display

    if payload.get('markdown'):
        display(Markdown(payload['markdown']))
    else:
        print('no rendered Markdown beside that artifact.')


def mirror_to_drive(sub: str = 'experiments') -> None:
    """Copy the VM's runs to Drive, minus what is rebuildable, so the session survives the machine."""
    where = ['--drive', ZTE_DRIVE, '--write-mode', WRITE_MODE, '--direction', 'up']
    payload = colab('mirror', *where, '--date', RUN_DATE, '--sub', sub)
    if reason := payload['skipped_reason']:
        print(f'nothing mirrored: {reason}')
    else:
        print(f'{payload["src"]} -> {payload["dst"]}   ({payload["copied"]} copied, {payload["failed"]} failed)')


def show_resources() -> None:
    """Prints RAM / GPU / disk as they stand now, so an out-of-memory kill is predictable rather than a mystery."""
    r = colab('env')['resources']
    gpu = f'{r["gpu"]["name"]} ({r["gpu"]["total_gb"]} GB)' if r['gpu'] else 'none'
    print(f'RAM {r["ram_gb"]} GB - {r["cpu_count"]} cores - {r["free_disk_gb"]} GB free disk - GPU {gpu}')


show_resources()

In [ ]:
# Rendering only: these read the JSON that `zte-colab` and the audit CLIs already produced.
import pandas as pd
import plotly.graph_objects as go

## 5 · Prepare the data once, on Drive, and never again

The prepared bundle is keyed by a full-config hash and staged on the roomiest volume the VM has. Re-running this is
a no-op once the bundle exists, and it is mirrored to Drive so the next session skips it entirely.

This is the only cell that touches the 17–23 GB raw archives.

In [ ]:
!uv run zte-prepare --root "{DATA_DIR}" --configs experiments/alignment --cache-dir "{PREPARED_LOCAL}" --remote "{PREPARED_DRIVE}"

## 6 · Experiment 1 — the granularity ablation

One contrastive term, moved between three units and nothing else changed:

| level | the unit it aligns | frozen target |
| --- | --- | --- |
| `sentence` | the pooled sentence vector | a frozen E5 sentence embedding |
| `word` | one fixated word = one EEG token | a frozen word vector |
| `token` | four fixed intra-word slices of one word | the LM's sub-word embeddings |

The claim under test is that finer alignment recovers more. The claim that has to be *excluded first* is that finer
alignment recovers more **spelling**.

### 6a · The floor, with no model involved

Run this before anything else. Four brain-free signatures, each scored as a retrieval oracle over your own
700-sentence gallery — no checkpoint, no training, nothing but the reference text.

| signature | what it is told, and nothing else |
| --- | --- |
| `words` | the word count |
| `total` | the total sub-word piece count, one integer per sentence |
| `multiset` | the piece counts with their order destroyed |
| `profile` | the ordered per-word piece counts |

`information_bits` is $\log_2 n - \frac{1}{n}\sum_i \log_2 m_i$, where $m_i$ counts the gallery sentences sharing
sentence *i*'s signature. On 700 sentences the ceiling is $\log_2 700 = 9.4512$ bits.

Watch **`alignment_coverage`** — the fraction of ZuCo words that matched their own reference text. Below about 0.99
the piece counts are partly wrong and the bits are not trustworthy.

In [ ]:
!uv run zte-audit --root "{DATA_DIR}" --piece-oracle --out "{SUITE}/audit/confound_audit.md"

In [ ]:
ORACLE = audit('oracle', f'{SUITE}/audit', markdown=False)['payload']['piece_oracle']

rows = [
    {
        'signature': name,
        'Top-1': round(block['top1'], 4),
        'hits/700': round(block['top1'] * block['n']),
        'bits of 9.4512': round(block['information_bits'], 3),
        'unique fraction': round(block['unique_fraction'], 3),
    }
    for name, block in ORACLE['oracles'].items()
]
display(pd.DataFrame(rows))

print(f'tokeniser : {ORACLE["tokenizer"]}')
print(
    f'coverage  : {ORACLE["alignment_coverage"]:.5f}'
    f'{"" if ORACLE["alignment_coverage"] > 0.99 else "   <-- below 0.99, the piece counts are partly wrong"}'
)
print(f'gate      : {ORACLE["gate_signature"]} at Top-1 {ORACLE["gate_top1"]:.4f} ({ORACLE["gate_bits"]:.2f} bits)')
print(f'ceiling   : {ORACLE["ceiling_signature"]} at Top-1 {ORACLE["ceiling_top1"]:.4f}')

**How to read that.** `gate` is the floor a *fixed* sub-token count can actually reach — what this repository builds,
because `objective.token_sub_tokens` is a constant per word and never the count of pieces the reference spells that
word in. `ceiling` is what a design that sized a word's EEG by its own piece count would have handed over.

The number to compare either against is the best held-out Top-1 this programme has measured. Section 6c prints it
beside the floor rather than leaving you to do the arithmetic.

### 6b · Train the three levels — optional, and expensive

Twelve folds × three levels is roughly 110 GPU-hours. **If the alignment arms already exist on Drive, skip this and
go straight to 6c**, which reads them. The planner below says what is already done, so a reclaimed VM resumes
exactly where it stopped.

In [ ]:
LEVELS = ['sentence', 'word', 'token']
PLAN = colab('sweep', 'status', '--levels', *LEVELS, '--out-root', OUT_ROOT, '--drive', ZTE_DRIVE)

for block in PLAN['progress']['tiers']:
    print(f'{block["tier"]:<11} {block["done"]:>3}/{block["total"]:<3} done  -  {block["hours_remaining"]:.1f} h left')

nxt = PLAN['next']
print()
print(f'next up: {nxt["run_name"]}  ->  {nxt["out_dir"]}' if nxt else 'nothing left to train at these levels.')

Each fold is one `zte-run` with `--resume`, which is idempotent and skips finished work. `FOLDS` lists all twelve
ZuCo subjects; trim it to shorten the sweep, but **a single holdout is not a population result** and section 13 is
where the mean ± sd comes from.

In [ ]:
FOLDS = ['ZAB', 'ZDM', 'ZDN', 'ZGW', 'ZJM', 'ZJN', 'ZJS', 'ZKB', 'ZKH', 'ZKW', 'ZMG', 'ZPH']

for level in LEVELS:
    for holdout in FOLDS:
        cfg = f'experiments/alignment/{level}/combined.yaml'
        !uv run zte-run --config {cfg} --root "{DATA_DIR}" --loso-holdout {holdout} --out-root "{OUT_ROOT}" --resume
    mirror_to_drive()

### 6c · The cross-level table, and the floor every level is read against

`zte-levels` reads the runs that already exist — it loads no model and re-scores no query — groups them by level,
aggregates across folds with a **sample** (n−1) standard deviation, and prints each level against its floor.

Two galleries are shown deliberately. The unstratified one is what a naive paper reports; the length-stratified one
is the honest cell, because on the unstratified gallery word count alone can answer the query.

In [ ]:
!uv run zte-levels --root "{OUT_ROOT}" --pattern "align_*" --out "{SUITE}/levels" --force

In [ ]:
show(audit('levels', f'{SUITE}/levels'))

In [ ]:
LEVELS_PAYLOAD = audit('levels', f'{SUITE}/levels', markdown=False)['payload']

fig = go.Figure()
names = [block['level'] for block in LEVELS_PAYLOAD['levels']]
strat = [(block.get('length_stratified') or {}) for block in LEVELS_PAYLOAD['levels']]
floors = [(block.get('length_floor') or {}).get('rank_percentile') for block in LEVELS_PAYLOAD['levels']]

fig.add_bar(
    x=names,
    y=[cell.get('rank_percentile') for cell in strat],
    error_y={'type': 'data', 'array': [cell.get('rank_percentile_sd') or 0 for cell in strat]},
    name='encoder (length-stratified)',
)
if any(floors):
    fig.add_hline(
        y=max(f for f in floors if f),
        line_dash='dash',
        line_color='crimson',
        annotation_text='length oracle (tol=1) - the brain-free floor',
        annotation_position='top left',
    )
fig.update_layout(
    title='Held-out rank percentile by alignment level, against the floor it must clear',
    yaxis_title='rank percentile',
    yaxis_range=[0.85, 1.0],
    height=420,
    showlegend=True,
)
fig.show()

**What this measures.** If every bar sits below the dashed line, the granularity comparison has no subject: the
levels are being ranked against each other inside a region a single integer already dominates. That is the measured
outcome on real ZuCo as of 2026-08-24, and `zte-levels` prints the confound-signature sentence rather than naming a
winner. A null is a finding, and this one is a *limit*: it says sub-word structure is not recoverable from
non-invasive EEG on this corpus, because any apparent recovery is bounded above by the spelling.

## 7 · Experiment 2 — the passage confound

The strongest objection to any cross-subject retrieval number is that the model memorised the passage rather than
read the meaning. ZuCo makes that objection testable, because the tasks' sentence sets are **disjoint**: no sentence
appears under both normal reading (NR) and sentiment reading (SR).

So train on one task and evaluate on another. An off-diagonal cell faces a **never-seen subject reading never-seen
stimuli**. If retrieval survives that, it is not passage memorisation.

`stimulus_novelty` measures the overlap rather than assuming it: a cell with any shared stimulus is flagged
`novel_stimuli: false` and cannot carry the claim.

In [ ]:
TASKS = ['NR', 'SR', 'TSR']

for task in TASKS:
    cfg = f'experiments/parallax/parallax_{task.lower()}.yaml'
    !uv run zte-run --config {cfg} --root "{DATA_DIR}" --loso-holdout ZAB --out-root "{OUT_ROOT}" --resume
mirror_to_drive()

In [ ]:
for train_task in TASKS:
    for eval_task in TASKS:
        ckpt = resolve_ckpt(f'parallax_{train_task.lower()}_loZAB_s42')
        !uv run zte-parallax transfer --ckpt "{ckpt}" --root "{DATA_DIR}" --eval-task {eval_task} --out "{SUITE}/transfer"

In [ ]:
!uv run zte-parallax report --transfers "{SUITE}/transfer" --out "{SUITE}/transfer"
show(audit('transfer', f'{SUITE}/transfer'))

**How to read the matrix.** Chance rank percentile is 0.5, so a confidence interval bracketing 0.5 is a null — and a
null here is a finding: it says the task-specific code did not transfer.

On the runs measured 2026-08-24, **NR→SR is the strongest cell in the matrix** at rank percentile 0.9595, above the
NR→NR diagonal at 0.9488, with 5 of 400 Top-1 hits at *p* = 0.0036. A model trained on normal reading reads
sentiment-reading sentences it has never seen, in a brain it has never seen, at least as well as it reads its own
task. **That is evidence against passage memorisation**, and it is the most positive result in this notebook.

Two qualifiers travel with it. Rank percentile resists the length confound; the Top-*k* on that cell does not, and 5
hits is below what settles a Top-*k* comparison. And TSR carries no measurable in-task content signal at all.

## 8 · Experiment 3 — the decoder reality check

This section is a **control experiment, not a headline.** Read it as the argument for why retrieval is the only
sound readout right now, rather than as a decoding result.

The arithmetic that makes it a control: the encoder supplies ~1.7–2.0 bits of sentence identity, a 19.6-word
sentence needs ~190, so free generation has about **1%** of what it requires. An honest null is the expected
outcome.

`zte-decode` runs the headline decode and then every pre-registered brain-independent control through the
*identical* `generate_from_prefix` path, so nothing differs between them but the conditioning:

| control | what it replaces | what it isolates |
| --- | --- | --- |
| `mean_prefix` | one prefix for every reading | anything the LM says regardless of input |
| `null_prefix` | no prefix at all | the LM's unconditional prior |
| `phase` | phase-scrambled EEG, power spectrum preserved | spectral shape without time structure |
| `noise` | noise-matched EEG at the **encoder input** | the encoder's response to matched noise |
| `noise_prefix` | noise-matched **z** straight into the bridge | how much of the readout is the LM's prior |
| `shuffled_z` | another reading's prefix | "any well-formed prefix" |
| `length_only` | a prefix built from word count alone | the 5.14-bit length channel |
| `mismatch` | a deliberately wrong pairing | the scoring itself |

`noise_prefix` is the one that answers the question directly. It is drawn with matched per-feature mean and variance
against the real **z** cloud — an off-manifold standard normal would be a trivially weak control and would let the
decoder look good for the wrong reason.

**The verdict gate ANDs over these**, and an unavailable or skipped control **fails** its clause. Adding a control
therefore tightens the gate, which is the intent.

In [ ]:
DECODE_CFG = 'experiments/flagship/decode_zte_v2.yaml'
DECODE_OUT = f'{SUITE}/decode'

!uv run zte-run --config {DECODE_CFG} --root "{DATA_DIR}" --out-root "{OUT_ROOT}" --resume

In [ ]:
DECODE_CKPT = resolve_ckpt('decode_zte_v2_loZAB_s42')
!uv run zte-decode --ckpt "{DECODE_CKPT}" --root "{DATA_DIR}" --split test --capacity --out "{DECODE_OUT}"

In [ ]:
READINGS = colab('readings', '--from', DECODE_OUT, '--rows', '8')

verdict = READINGS['verdict']
print(f'generation_above_controls : {verdict["above_controls"]}')
print(f'worst control             : {verdict.get("worst_control")}  at  {verdict.get("worst_ci")}')
print(f'permutation p             : {verdict.get("permutation_p")}')
print(f'prefix-influence KL       : {verdict.get("prefix_kl")}  (floor {verdict.get("min_prefix_kl")})')
absent = verdict.get('controls_absent') or []
print(f'controls absent (= failing): {", ".join(absent) if absent else "none"}')
print()
for name, passed in (verdict.get('clauses') or {}).items():
    print(f'  {"PASS" if passed else "FAIL"}  {name}')

In [ ]:
frame = pd.DataFrame(
    [
        {'condition': cond['name'], **{k: round(v, 4) for k, v in (cond['scores'] or {}).items()}}
        for cond in READINGS['readings'][0]['conditions']
    ]
)
display(frame)

print('\nOne reading, every condition. If the noise_prefix row scores like the hypothesis row on content')
print('metrics, the decode is the language model, not the brain.')

**What a null here establishes.** If the EEG-conditioned decode does not beat `noise_prefix` on content metrics,
then content-metric scores of generated text are not a measure of neural decoding — they are a measure of the
language model's prior plus whatever length leaks through. That is a methodological claim about the field's
evaluation practice, and it is worth more than a weak positive.

It is also exactly why this project reports the powered readout as **decoder-rescoring retrieval over the
700-sentence gallery, and never as generation**. `verdict['generation_above_controls']` is the only gate that would
license a generation headline, and it must be `True` on an honest split with every control beaten.

## 9 · Experiment 4 — the anchor-calibration curve

**The deployment question.** A new reader arrives. They read *N* sentences whose text is known. With **no
retraining** — the encoder stays frozen, only a per-subject map into the shared space is fitted — how much does
their retrieval improve, and does it improve at all?

Three arms are scored at every anchor count, on one identical gallery:

| arm | what it is | what it controls for |
| --- | --- | --- |
| `uncalibrated` | frozen embeddings, anchors removed | the gallery got smaller, so retrieval got easier for free |
| `calibrated` | the map fitted on true (reading, text) pairs | the measurement |
| `shuffled` | the same map fitted on a **derangement** of those pairs | the transform's raw capacity |

**The `shuffled` arm is the one that makes this believable.** It fits the same map with the same parameter count on
the same number of pairs, with every anchor paired to the *wrong* reference. Whatever it lifts is what the transform
buys by existing rather than by being calibrated. A curve that does not beat its own shuffled control is not a
calibration result.

Two families bracket what any affine calibration could buy: `procrustes` is rotation-only, so it cannot inflate a
number by rescaling; `ridge` is strictly more expressive.

**Every anchor stimulus is removed from the query set *and* the gallery.** Leaving one in the gallery would let the
map place a sentence it was fitted on next to itself and manufacture the whole effect. That is why anchor count 0 is
re-scored on each reduced gallery rather than once on the full one — the columns you compare are the same problem.

Read `docs/CALIBRATION.md` for the algebra. Two things to know before the curve appears:

- The often-quoted "+0.0628 lift from 12 shared words" is a **cohesion** diagnostic at *word* level whose fitted map
  was never applied to a scored embedding. It is not evidence that retrieval moves.
- `dataset.raw_align_fit` defaults to `'all'`, so unlabelled per-subject whitening has **already** happened on the
  holdout. This curve measures what *labelled* anchors add on top of that.

In [ ]:
CALIB_CKPT = resolve_ckpt('align_sentence_combined_loZAB_s42')
CALIB_OUT = f'{SUITE}/calibration'

!uv run zte-calibrate --ckpt "{CALIB_CKPT}" --root "{DATA_DIR}" --anchor-counts 0,10,25,50,100,200 --draws 5 --family both --out "{CALIB_OUT}"

In [ ]:
show(audit('calibration', CALIB_OUT))

In [ ]:
CALIB = audit('calibration', CALIB_OUT, markdown=False)['payload']
GALLERY = CALIB.get('headline_gallery', 'length_stratified')

fig = go.Figure()
for family, galleries in (CALIB.get('series') or {}).items():
    series = galleries.get(GALLERY) or {}
    x = series.get('anchor_counts') or []
    for key, label, dash in (
        ('calibrated_rank_percentile', f'{family} - calibrated', 'solid'),
        ('shuffled_rank_percentile', f'{family} - shuffled anchors (control)', 'dot'),
    ):
        if series.get(key):
            fig.add_scatter(x=x, y=series[key], mode='lines+markers', name=label, line={'dash': dash})
    if series.get('uncalibrated_rank_percentile'):
        fig.add_scatter(
            x=x,
            y=series['uncalibrated_rank_percentile'],
            mode='lines',
            name=f'{family} - uncalibrated, same gallery',
            line={'dash': 'dash'},
        )

fig.update_layout(
    title=f'Held-out rank percentile vs. calibration anchors ({GALLERY} gallery)',
    xaxis_title='labelled sentences from the new reader',
    yaxis_title='rank percentile',
    height=460,
)
fig.show()

for family, block in (CALIB.get('verdict') or {}).items():
    print(
        f'{family:<11} lift {block.get("lift")}  vs shuffled {block.get("shuffled_lift")}  '
        f'->  helps={block.get("helps")}  beats_shuffled={block.get("beats_shuffled")}'
    )
    print(f'            {block.get("verdict")}')

**How to read the curve.** The gap that matters is **calibrated minus shuffled**, not calibrated minus zero. The
report gives it as a *paired* per-draw difference with an interval (`margin_over_shuffled_ci`), because which
particular 10 sentences a reader happened to get is a real source of variance and every point is repeated over
seeded draws.

Three fields decide whether a point may be quoted at all, and the rendered report prints all three:
`underdetermined` (fewer anchors than dimensions — a `ridge` map at *N* = 10 in 768 dimensions is interpolating),
`degraded_fits` (draws that failed to fit; the fitter returns `None` and logs rather than silently becoming an
identity), and `saturated` (the reader had fewer stimuli than the count requested).

If this curve rises and beats its shuffled control, ZTE has a rapid-deployment property that needs no retraining —
the single most useful thing a BCI can have. If it does not, that is a deployment limit worth stating plainly.

## 10 · Experiment 5 — length- and piece-matched semantic hard negatives

The idea: punish the encoder for the *lazy semantic mistake*. If the target is "the dog bit the man", the negative
should be "the man bit the dog" — same length, same piece profile, same vocabulary, different meaning. Separate
those and the win cannot be surface form.

The repository already mined negatives by `surface_overlap − semantic_cosine`. That is the right ranking and the
wrong candidate set: it imposed **no length constraint**, so on this corpus a mined negative can still be told apart
by counting words, which teaches the encoder nothing. Since word count carries 5.14 of the 9.45 bits, matching on
length and on the sub-word piece budget is what makes a hard negative actually hard.

`hard_negative_strategy` selects the candidate set, and `hard_negative_in_loss` decides whether the mined table
narrows the **loss denominator** or only the batch composition. Both default off, so every existing run stays
byte-identical.

The pair below flips exactly those knobs against the measured sentence arm — one lever, so the difference is
attributable.

In [ ]:
HARDNEG_ARMS = [
    ('experiments/alignment/sentence/combined.yaml', 'the measured baseline'),
    ('experiments/alignment/sentence/hardneg.yaml', 'piece-matched hard negatives, in the loss'),
]

for cfg, label in HARDNEG_ARMS:
    print(f'\n===== {label} =====')
    !uv run zte-run --config {cfg} --root "{DATA_DIR}" --loso-holdout ZAB --out-root "{OUT_ROOT}" --resume
mirror_to_drive()

In [ ]:
for run in ('align_sentence_combined_loZAB_s42', 'align_sentence_hardneg_loZAB_s42'):
    ckpt = resolve_ckpt(run)
    !uv run zte-rebaseline --ckpt "{ckpt}" --root "{DATA_DIR}" --piece-oracle --out "{SUITE}/hardneg/{run}"

In [ ]:
rows = []
for run in ('align_sentence_combined_loZAB_s42', 'align_sentence_hardneg_loZAB_s42'):
    payload = audit('rebaseline', f'{SUITE}/hardneg/{run}', markdown=False)['payload']
    if not payload:
        continue
    honest = payload['grid']['train_fitted']['length_stratified']
    floor = payload['floor_comparison']
    row = {'arm': run, 'rank pct': round(honest['rank_percentile'], 4)}
    row['CI'] = [round(v, 4) for v in honest['rank_percentile_ci'][1:]]
    row['Top-1 hits'] = round(honest['top1'] * honest['n_queries'])
    row['of'] = honest['n_queries']
    row['p'] = f'{honest["top1_p"]:.3g}'
    row['length floor'] = round(floor['oracle'], 4)
    row['clears'] = floor['clears_floor']
    rows.append(row)

display(pd.DataFrame(rows))

**What would count as a win.** Not a higher Top-1 on the unstratified gallery — that is where length lives. A win is
a higher **length-stratified rank percentile** whose interval clears the floor, on a matched pair that differs only
in the negative-mining strategy. Anything less is a different number, not a better model.

## 11 · Experiment 6 — the architecture benchmark

An ablation against a band-power MLP does not answer "is the conformer the right architecture". The established EEG
deep-learning baselines do, and running them through the *identical* InfoNCE pipeline is what makes the comparison
fair: the objective never sees the frontend, so only the encoder changes.

| arm | frontend | why it is here |
| --- | --- | --- |
| `align_sentence_combined` | `raw_conformer` | the recipe under test |
| `eegnet_clip` | `eegnet` | the compact depthwise-separable standard |
| `deepconvnet_clip` | `deep_conv_net` | the deeper convolutional standard |

Two design notes that are in the code rather than in a footnote:

- **DeepConvNet's canonical four max-pool blocks are tuned for 1000+ samples.** The live `raw_window` is 350 (700 ms
  at 500 Hz) and archived configs use 128. The pooling schedule is adapted to the actual window and raises a clear
  error rather than producing a zero-length time axis.
- **EEGNet's depthwise convolution already *is* a spatial filter over electrodes.** Stacking spherical-harmonic
  spatial encoding in front of it double-counts the geometry, so if both are on the run logs a warning. Treat that
  combination as an explicit ablation, never a default.

In [ ]:
BENCH_ARMS = [
    'experiments/benchmark/eegnet_clip.yaml',
    'experiments/benchmark/deepconvnet_clip.yaml',
]

for cfg in BENCH_ARMS:
    !uv run zte-run --config {cfg} --root "{DATA_DIR}" --loso-holdout ZAB --out-root "{OUT_ROOT}" --resume
mirror_to_drive()

In [ ]:
# A run directory is named `<run_name>_lo<subject>_s<seed>`, from the `run_name` inside the YAML.
BENCH_RUNS = [
    'align_sentence_combined_loZAB_s42',
    'benchmark_eegnet_clip_loZAB_s42',
    'benchmark_deepconvnet_clip_loZAB_s42',
]

for run in BENCH_RUNS:
    ckpt = resolve_ckpt(run)
    !uv run zte-rebaseline --ckpt "{ckpt}" --root "{DATA_DIR}" --piece-oracle --out "{SUITE}/benchmark/{run}"

In [ ]:
rows = []
for run in BENCH_RUNS:
    payload = audit('rebaseline', f'{SUITE}/benchmark/{run}', markdown=False)['payload']
    if not payload:
        continue
    honest = payload['grid']['train_fitted']['length_stratified']
    floor = payload['floor_comparison']
    row = {'arm': run, 'frontend': payload['provenance'].get('frontend')}
    row['rank pct'] = round(honest['rank_percentile'], 4)
    row['CI'] = [round(v, 4) for v in honest['rank_percentile_ci'][1:]]
    row['Top-1 hits'] = round(honest['top1'] * honest['n_queries'])
    row['of'] = honest['n_queries']
    row['length floor'] = round(floor['oracle'], 4)
    row['clears'] = floor['clears_floor']
    row['bits from EEG'] = round(payload['bit_budget']['bits_from_eeg'], 2)
    rows.append(row)

display(pd.DataFrame(rows))

**The honest framing of this table.** If no arm clears the floor, the benchmark does not say the conformer is better
or worse than EEGNet — it says the readout is confound-bound for all three, and the architecture is not the binding
constraint. That is a more useful statement to the field than a ranking inside the noise, and it is the outcome the
rest of this notebook's evidence predicts.

## 12 · Experiment 7 — where and when the readout comes from

### Why this is occlusion and not attention

The natural ask is "plot the conformer's temporal attention and show it peaks near 400 ms, the N400". That cannot be
done on any checkpoint that exists. `RawConformer` builds its attentive temporal pool only when
`conformer_temporal_pool: attention`, and **no live config sets it** — every flagship, decoder, parallax and
alignment arm uses mean pooling, so those weights do not exist in a trained model. The inner transformer is called
with `need_weights=False` and there are no forward hooks anywhere in the package.

Retraining an arm with attentive pooling would produce a *different model*, which could not then be compared to the
mean-pool runs as if it were the same one.

So this measures **causal contribution** instead: occlude a time span of the raw window, re-embed, and measure how
far the sentence vector moves. That works on every existing checkpoint, and it is the better instrument regardless —
attention weights are famously not explanations, whereas an occlusion drop is a counterfactual.

A **null band** travels with the profile: a same-width occlusion at a random offset, so the curve has a floor rather
than being a bare bar chart of drops.

In [ ]:
LENS_CKPT = resolve_ckpt('align_sentence_combined_loZAB_s42')
!uv run zte-lens encode --ckpt "{LENS_CKPT}" --root "{DATA_DIR}" --temporal --temporal-bins 14 --temporal-sentences 12 --out "{SUITE}/lens"

In [ ]:
show(audit('temporal', f'{SUITE}/lens'))

In [ ]:
TEMPORAL = audit('temporal', f'{SUITE}/lens', markdown=False)['payload']
bins = (TEMPORAL or {}).get('bins') or []

if bins:
    null_band = TEMPORAL.get('null_band') or {}
    fig = go.Figure()
    fig.add_scatter(
        x=[b['center_ms'] for b in bins],
        y=[b['mean_drop'] for b in bins],
        error_y={
            'type': 'data',
            'array': [b['ci_high'] - b['mean_drop'] for b in bins],
            'arrayminus': [b['mean_drop'] - b['ci_low'] for b in bins],
        },
        mode='lines+markers',
        name='occlusion drop',
    )
    if null_band.get('mean_drop') is not None:
        fig.add_hline(
            y=null_band['mean_drop'],
            line_dash='dot',
            line_color='grey',
            annotation_text='random-offset null band',
            annotation_position='top left',
        )
    lo, hi = TEMPORAL.get('n400_window_ms', [300, 500])
    fig.add_vrect(
        x0=lo,
        x1=hi,
        fillcolor='orange',
        opacity=0.12,
        line_width=0,
        annotation_text=f'{lo}-{hi} ms',
        annotation_position='top right',
    )
    fig.update_layout(
        title='Causal contribution by latency within the word window',
        xaxis_title='ms from word onset',
        yaxis_title='cosine drop when occluded',
        height=440,
    )
    fig.show()

    peak = TEMPORAL.get('peak') or {}
    print(f'window            : {TEMPORAL.get("window_ms")} ms over {TEMPORAL.get("raw_window_samples")} samples')
    print(f'readings / words  : {TEMPORAL.get("n_readings")} / {TEMPORAL.get("n_words")}')
    print(f'peak bin          : {peak.get("start_ms")}-{peak.get("end_ms")} ms   above null: {peak.get("above_null")}')
    print(f'peak in {TEMPORAL.get("n400_window_ms")} ms : {TEMPORAL.get("peak_in_n400_window")}')
    print(f'\n{TEMPORAL.get("caveat", "")}')
else:
    print('No temporal profile. This instrument needs a raw-input checkpoint; a band-power run writes none.')

**The claim this does *not* license.** ZuCo word windows come from eye-tracking segmentation and overlap their
neighbours, so a peak in 300–500 ms is *consistent with* an N400 and is not proof of one. `peak_in_n400_window`
gates nothing, and the profile carries the lens disclaimer: it is an inspection surface, not a result.

### The scalp side, and why it needs a real montage first

A topographic map of channel importance is only interpretable if the channel axis maps to real electrode positions.
**Every live configuration in this repository runs on the approximate Fibonacci-cap geometry** — no config sets
`dataset.montage_csv`, so `resolve_geometry` falls back, logs it, and sets `approximate=True` as a *persistent*
buffer inside the checkpoint.

The mathematics is unaffected (the fallback is a genuine set of well-separated points on the sphere, so rotation
structure and the addition theorem hold exactly for those points). What is lost is the correspondence to a head: a
topoplot from such a run shows which **array indices** mattered, not which brain regions. Read the checkpoint's flag,
never the YAML — `resolve_geometry` also swallows a malformed CSV and silently falls back.

So export the real GSN-HydroCel montage first, then train an arm that points at it. Until that arm exists, the
channel map below is a picture of indices and is labelled as such.

In [ ]:
# `--spatial exact` builds the ZuCo-105 montage through mne and wires it into the run, so the checkpoint records
# real coordinates rather than the Fibonacci fallback. It needs `uv sync --group spatial`, which section 1 did.
!uv run zte-run --config experiments/alignment/sentence/combined.yaml --root "{DATA_DIR}" \
    --loso-holdout ZAB --spatial exact --montage-out res/montage_gsn105.csv \
    --name align_sentence_montage --out-root "{OUT_ROOT}" --resume

In [ ]:
MONTAGE_CKPT = resolve_ckpt('align_sentence_montage')
!uv run zte-lens encode --ckpt "{MONTAGE_CKPT}" --root "{DATA_DIR}" --html --out "{SUITE}/lens_montage"

The lens writes channel saliency into `lens.json` and, with `--html`, a `LENS.html` page carrying the scalp map. A
map from the montage-wired checkpoint above is positionally meaningful; one from any other arm in this repository is
not. **The condition for a regional claim is the checkpoint's own `approximate_geometry` flag reading `False`** —
never the YAML, because `resolve_geometry` swallows a malformed CSV and silently falls back to the cap.

## 13 · Experiment 8 — the full twelve-fold LOSO, with statistics

A single held-out subject is a sample of one. Nothing in this notebook may be quoted as a population result without
this section, and reviewers will not accept it otherwise.

`zte-loso-summary` aggregates the folds. Two things about what it reports:

- The headline is the **rank percentile**, aggregated across folds — not the per-fold pooled Top-1 that appears in a
  session `INDEX.md`, which is the inflated number.
- The spread is a **sample** (n−1) standard deviation. At *n* = 12 the population form under-reports it by about 4%,
  which matters when the spread is the finding.

Section 6b already trains the twelve folds if you ran it. This section reads them.

In [ ]:
# `--out` is the Markdown path; a `.csv` of the same stem is written beside it.
!uv run zte-loso-summary --experiments "{OUT_ROOT}" --out "{SUITE}/loso/LOSO_SUMMARY.md"

In [ ]:
from IPython.display import Markdown, display

summary = pathlib.Path(f'{SUITE}/loso/LOSO_SUMMARY.md')
if summary.is_file():
    display(Markdown(summary.read_text()))
    display(pd.read_csv(summary.with_suffix('.csv')))
else:
    print(f'no LOSO summary at {summary}; train the folds in section 6b first.')

**Read the held-out lift, and read it beside the floor.** A positive lift over chance on the unstratified gallery is
not the same claim as clearing the length oracle. Section 14 puts both in one table so the two cannot be confused.

## 14 · The evidence board

One command reads every artifact this notebook produced and assembles the claims. It **recomputes nothing** — each
row is read from the audit that wrote it, so the board cannot disagree with the runs it describes.

Three properties make it airtight:

1. **A claim with no floor is `not measured`, never a result.** There is no code path by which an unfloored number
   becomes a headline.
2. **The interval must clear the floor, not the point estimate.** A point estimate above a floor with an interval
   straddling it is the shape every retracted result in this project had.
3. **A missing artifact is named, not dropped.** A silently absent row would read as a claim nobody made — which is
   how a gap becomes an implied pass.

The board's own gates are mutation-tested: break the floor comparison and the test suite goes red.

In [ ]:
!uv run zte-evidence --roots "{SUITE}" --title "ZTE evidence board - {RUN_DATE}" --out "{SUITE}/board" --force

In [ ]:
show(audit('evidence', f'{SUITE}/board'))

In [ ]:
BOARD = audit('evidence', f'{SUITE}/board', markdown=False)['payload']

rows = []
for row in BOARD['claims']:
    cell = {'claim': row['key'], 'verdict': row['verdict']}
    cell['value'] = None if row['value'] is None else round(row['value'], 4)
    cell['CI'] = None if not row['ci'] else [round(v, 4) for v in row['ci']]
    cell['floor'] = None if row['floor'] is None else round(row['floor'], 4)
    cell['floor is'] = row['floor_name']
    cell['quotable alone'] = row['headline_safe']
    rows.append(cell)

display(pd.DataFrame(rows))

print(f'\n{BOARD["n_headline_safe"]} of {len(BOARD["claims"])} claim(s) may be quoted without the floor sentence.')
for key, why in sorted(BOARD['missing'].items()):
    print(f'  not measured: {key} - {why}')

### What to do with this

**If a row is green**, it clears a brain-free floor with its interval and may be quoted on its own. Say which floor,
which gallery, and which `postprocess_fit` alongside it.

**If every row is below its floor**, that is the result, and it is publishable: it establishes a *resolution limit*
for non-invasive EEG on this corpus rather than a failure of one model. The field currently reports token- and
word-level decoding numbers on galleries where spelling resolves almost every item; a measured floor that no
architecture clears is the corrective standard, and the piece oracle is the instrument that makes it checkable by
anyone.

**If a row is `not measured`**, run the section that produces it before writing about it.

State the null plainly. On work aimed at people with ALS and locked-in syndrome, an overclaimed result is worse than
no result.

## 15 · Persist, resume, continue

Everything above already lives on Drive. This snapshots the session so a fresh VM — or a fresh month — picks it up
unchanged. Every long cell is resumable and `--resume` is idempotent, so a reclaimed runtime costs only the cell it
was inside.

In [ ]:
mirror_to_drive()
show_resources()

print(f'\nsuite    : {SUITE}')
print(f'board    : {SUITE}/board/evidence.md')
print(f'session  : {DRIVE_DIR}')
print(f'\nTo resume on a new VM: set RESUME_DATE = {RUN_DATE!r} in section 4, then run from section 1.')